# ViT

> Pretrained ViT (via timm) and from scratch implementations (small, and large)


In [ ]:
#| default_exp models.vision.vits

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
# https://github.com/lucidrains/vit-pytorch/blob/main/vit_pytorch/vit.py

import torch
from torch import nn

from einops import rearrange, repeat
from einops.layers.torch import Rearrange


## Standard ViT

In [ ]:
#| export

def pair(t):
    return t if isinstance(t, tuple) else (t, t)

# classes

class PreNorm(nn.Module):
    def __init__(self, dim, fn):
        super().__init__()
        self.norm = nn.LayerNorm(dim)
        self.fn = fn
    def forward(self, x, **kwargs):
        return self.fn(self.norm(x), **kwargs)

class FeedForward(nn.Module):
    def __init__(self, dim, hidden_dim, dropout = 0.):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(dim, hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, dim),
            nn.Dropout(dropout)
        )
    def forward(self, x):
        return self.net(x)

class Attention(nn.Module):
    def __init__(self, dim, heads = 8, dim_head = 64, dropout = 0.):
        super().__init__()
        inner_dim = dim_head *  heads
        project_out = not (heads == 1 and dim_head == dim)

        self.heads = heads
        self.scale = dim_head ** -0.5

        self.attend = nn.Softmax(dim = -1)
        self.to_qkv = nn.Linear(dim, inner_dim * 3, bias = False)

        self.to_out = nn.Sequential(
            nn.Linear(inner_dim, dim),
            nn.Dropout(dropout)
        ) if project_out else nn.Identity()

    def forward(self, x):
        qkv = self.to_qkv(x).chunk(3, dim = -1)
        q, k, v = map(lambda t: rearrange(t, 'b n (h d) -> b h n d', h = self.heads), qkv)

        dots = torch.matmul(q, k.transpose(-1, -2)) * self.scale

        attn = self.attend(dots)

        out = torch.matmul(attn, v)
        out = rearrange(out, 'b h n d -> b n (h d)')
        return self.to_out(out)

class Transformer(nn.Module):
    def __init__(self, dim, depth, heads, dim_head, mlp_dim, dropout = 0.):
        super().__init__()
        self.layers = nn.ModuleList([])
        for _ in range(depth):
            self.layers.append(nn.ModuleList([
                PreNorm(dim, Attention(dim, heads = heads, dim_head = dim_head, dropout = dropout)),
                PreNorm(dim, FeedForward(dim, mlp_dim, dropout = dropout))
            ]))
    def forward(self, x):
        for attn, ff in self.layers:
            x = attn(x) + x
            x = ff(x) + x
        return x


In [ ]:
#| export

class ViTBackbone(nn.Module):
    def __init__(self, *, img_size, patch_size, hidden_dim, depth, heads, mlp_dim, pool = 'cls', dim_head = 64, dropout = 0., emb_dropout = 0.):
        super().__init__()
        channels = img_size[0]
        img_size = img_size[1]
        image_height, image_width = pair(img_size)
        patch_height, patch_width = pair(patch_size)

        assert image_height % patch_height == 0 and image_width % patch_width == 0, 'Image dimensions must be divisible by the patch size.'

        num_patches = (image_height // patch_height) * (image_width // patch_width)
        patch_dim = channels * patch_height * patch_width
        assert pool in {'cls', 'mean'}, 'pool type must be either cls (cls token) or mean (mean pooling)'

        self.to_patch_embedding = nn.Sequential(
            Rearrange('b c (h p1) (w p2) -> b (h w) (p1 p2 c)', p1 = patch_height, p2 = patch_width),
            nn.Linear(patch_dim, hidden_dim),
        )

        self.pos_embedding = nn.Parameter(torch.randn(1, num_patches + 1, hidden_dim))
        self.cls_token = nn.Parameter(torch.randn(1, 1, hidden_dim))
        self.dropout = nn.Dropout(emb_dropout)

        self.transformer = Transformer(hidden_dim, depth, heads, dim_head, mlp_dim, dropout)

        self.pool = pool
        self.to_latent = nn.Identity()

    def forward(self, img):
        x = self.to_patch_embedding(img)
        b, n, _ = x.shape

        cls_tokens = repeat(self.cls_token, '() n d -> b n d', b = b)
        x = torch.cat((cls_tokens, x), dim=1)
        x += self.pos_embedding[:, :(n + 1)]
        x = self.dropout(x)

        x = self.transformer(x)

        x = x.mean(dim = 1) if self.pool == 'mean' else x[:, 0]

        x = self.to_latent(x)
        return x

In [ ]:
#| export
class ViT(nn.Module):
    def __init__(self, img_size, patch_size, num_classes, hidden_dim, depth, heads, mlp_dim, pool = 'cls', dim_head = 64, dropout = 0., emb_dropout = 0.):
        super().__init__()

        self.backbone = ViTBackbone(img_size = img_size, patch_size = patch_size, hidden_dim= hidden_dim, depth= depth, heads= heads, mlp_dim= mlp_dim, pool = pool, dim_head = dim_head, dropout = dropout, emb_dropout = emb_dropout)  
        self.head = nn.Sequential(
            nn.LayerNorm(hidden_dim),
            nn.Linear(hidden_dim, num_classes)
        )

    def forward(self, x):
        h = self.backbone(x)
        logits = self.head(h)
        return logits

In [ ]:
#| hide
model = ViT(img_size = (3, 32, 32), patch_size = 4, num_classes = 10, hidden_dim= 256, depth= 6, heads= 8, mlp_dim= 1024)
model

ViT(
  (backbone): ViTBackbone(
    (to_patch_embedding): Sequential(
      (0): Rearrange('b c (h p1) (w p2) -> b (h w) (p1 p2 c)', p1=4, p2=4)
      (1): Linear(in_features=48, out_features=256, bias=True)
    )
    (dropout): Dropout(p=0.0, inplace=False)
    (transformer): Transformer(
      (layers): ModuleList(
        (0-5): 6 x ModuleList(
          (0): PreNorm(
            (norm): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
            (fn): Attention(
              (attend): Softmax(dim=-1)
              (to_qkv): Linear(in_features=256, out_features=1536, bias=False)
              (to_out): Sequential(
                (0): Linear(in_features=512, out_features=256, bias=True)
                (1): Dropout(p=0.0, inplace=False)
              )
            )
          )
          (1): PreNorm(
            (norm): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
            (fn): FeedForward(
              (net): Sequential(
                (0): Linear(in_features

In [ ]:
#| hide
inp = torch.randn(1, 3, 32, 32)
model(inp).shape
model.backbone(inp).shape

torch.Size([1, 256])

## Small ViT

In [ ]:
#| export

from math import sqrt
import torch
import torch.nn.functional as F
from torch import nn

from einops import rearrange, repeat
from einops.layers.torch import Rearrange


In [ ]:
#| export
class LSA(nn.Module):
    def __init__(self, dim, heads = 8, dim_head = 64, dropout = 0.):
        super().__init__()
        inner_dim = dim_head *  heads
        self.heads = heads
        self.temperature = nn.Parameter(torch.log(torch.tensor(dim_head ** -0.5)))

        self.attend = nn.Softmax(dim = -1)
        self.to_qkv = nn.Linear(dim, inner_dim * 3, bias = False)

        self.to_out = nn.Sequential(
            nn.Linear(inner_dim, dim),
            nn.Dropout(dropout)
        )

    def forward(self, x):
        qkv = self.to_qkv(x).chunk(3, dim = -1)
        q, k, v = map(lambda t: rearrange(t, 'b n (h d) -> b h n d', h = self.heads), qkv)

        dots = torch.matmul(q, k.transpose(-1, -2)) * self.temperature.exp()

        mask = torch.eye(dots.shape[-1], device = dots.device, dtype = torch.bool)
        mask_value = -torch.finfo(dots.dtype).max
        dots = dots.masked_fill(mask, mask_value)

        attn = self.attend(dots)

        out = torch.matmul(attn, v)
        out = rearrange(out, 'b h n d -> b n (h d)')
        return self.to_out(out)

class Transformer(nn.Module):
    def __init__(self, dim, depth, heads, dim_head, mlp_dim, dropout = 0.):
        super().__init__()
        self.layers = nn.ModuleList([])
        for _ in range(depth):
            self.layers.append(nn.ModuleList([
                PreNorm(dim, LSA(dim, heads = heads, dim_head = dim_head, dropout = dropout)),
                PreNorm(dim, FeedForward(dim, mlp_dim, dropout = dropout))
            ]))
    def forward(self, x):
        for attn, ff in self.layers:
            x = attn(x) + x
            x = ff(x) + x
        return x

class SPT(nn.Module):
    def __init__(self, *, dim, patch_size, channels = 3):
        super().__init__()
        patch_dim = patch_size * patch_size * 5 * channels

        self.to_patch_tokens = nn.Sequential(
            Rearrange('b c (h p1) (w p2) -> b (h w) (p1 p2 c)', p1 = patch_size, p2 = patch_size),
            nn.LayerNorm(patch_dim),
            nn.Linear(patch_dim, dim)
        )

    def forward(self, x):
        shifts = ((1, -1, 0, 0), (-1, 1, 0, 0), (0, 0, 1, -1), (0, 0, -1, 1))
        shifted_x = list(map(lambda shift: F.pad(x, shift), shifts))
        x_with_shifts = torch.cat((x, *shifted_x), dim = 1)
        return self.to_patch_tokens(x_with_shifts)


In [ ]:
#| export
class ViTSmallBackbone(nn.Module):
    def __init__(self, *, img_size, patch_size, hidden_dim, depth, heads, mlp_dim, pool = 'cls', channels = 3, dim_head = 64, dropout = 0., emb_dropout = 0.):
        super().__init__()
        image_height, image_width = pair(img_size)
        patch_height, patch_width = pair(patch_size)

        assert image_height % patch_height == 0 and image_width % patch_width == 0, 'Image dimensions must be divisible by the patch size.'

        num_patches = (image_height // patch_height) * (image_width // patch_width)
        patch_dim = channels * patch_height * patch_width
        assert pool in {'cls', 'mean'}, 'pool type must be either cls (cls token) or mean (mean pooling)'

        self.to_patch_embedding = SPT(dim = hidden_dim, patch_size = patch_size, channels = channels)

        self.pos_embedding = nn.Parameter(torch.randn(1, num_patches + 1, hidden_dim))
        self.cls_token = nn.Parameter(torch.randn(1, 1, hidden_dim))
        self.dropout = nn.Dropout(emb_dropout)

        self.transformer = Transformer(hidden_dim, depth, heads, dim_head, mlp_dim, dropout)

        self.pool = pool
        self.to_latent = nn.Identity()

    def forward(self, img):
        x = self.to_patch_embedding(img)
        b, n, _ = x.shape

        cls_tokens = repeat(self.cls_token, '() n d -> b n d', b = b)
        x = torch.cat((cls_tokens, x), dim=1)
        x += self.pos_embedding[:, :(n + 1)]
        x = self.dropout(x)

        x = self.transformer(x)

        x = x.mean(dim = 1) if self.pool == 'mean' else x[:, 0]

        x = self.to_latent(x)
        return x

In [ ]:
#| export
class ViTSmall(nn.Module):
    def __init__(self, img_size, patch_size, num_classes, hidden_dim, depth, heads, mlp_dim, pool = 'cls', dim_head = 64, dropout = 0., emb_dropout = 0.):
        super().__init__()
        channels = img_size[0]
        img_size = img_size[1]
        self.backbone = ViTSmallBackbone(img_size = img_size, patch_size = patch_size, hidden_dim= hidden_dim, depth= depth, heads= heads, mlp_dim= mlp_dim, pool = pool, channels = channels, dim_head = dim_head, dropout = dropout, emb_dropout = emb_dropout)  
        self.head = nn.Sequential(
            nn.LayerNorm(hidden_dim),
            nn.Linear(hidden_dim, num_classes)
        )

    def forward(self, x):
        h = self.backbone(x)
        logits = self.head(h)
        return logits

In [ ]:
#| hide
model = ViTSmall(img_size = (3, 32, 32), patch_size = 4, num_classes = 10, hidden_dim= 256, depth= 6, heads= 8, mlp_dim= 1024)
model

ViTSmall(
  (backbone): ViTSmallBackbone(
    (to_patch_embedding): SPT(
      (to_patch_tokens): Sequential(
        (0): Rearrange('b c (h p1) (w p2) -> b (h w) (p1 p2 c)', p1=4, p2=4)
        (1): LayerNorm((240,), eps=1e-05, elementwise_affine=True)
        (2): Linear(in_features=240, out_features=256, bias=True)
      )
    )
    (dropout): Dropout(p=0.0, inplace=False)
    (transformer): Transformer(
      (layers): ModuleList(
        (0-5): 6 x ModuleList(
          (0): PreNorm(
            (norm): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
            (fn): LSA(
              (attend): Softmax(dim=-1)
              (to_qkv): Linear(in_features=256, out_features=1536, bias=False)
              (to_out): Sequential(
                (0): Linear(in_features=512, out_features=256, bias=True)
                (1): Dropout(p=0.0, inplace=False)
              )
            )
          )
          (1): PreNorm(
            (norm): LayerNorm((256,), eps=1e-05, elementwise_af

In [ ]:
#| hide
model(torch.randn(1, 3, 32, 32)).shape
model.backbone(torch.randn(1, 3, 32, 32)).shape

torch.Size([1, 256])

### Pretrained ViT (via timm vit_base_patch16_384)

In [ ]:
#| export
import timm
import torch
import torch.nn as nn

In [ ]:
#| export
def timm_vit_model(model_id= "vit_small_patch8_224", pretrained= True):
    model = timm.create_model(model_id, pretrained= pretrained, drop_path_rate=0.1)
    return model

In [ ]:
#| hide
model = timm_vit_model()
model

VisionTransformer(
  (patch_embed): PatchEmbed(
    (proj): Conv2d(3, 384, kernel_size=(8, 8), stride=(8, 8))
    (norm): Identity()
  )
  (pos_drop): Dropout(p=0.0, inplace=False)
  (patch_drop): Identity()
  (norm_pre): Identity()
  (blocks): Sequential(
    (0): Block(
      (norm1): LayerNorm((384,), eps=1e-06, elementwise_affine=True)
      (attn): Attention(
        (qkv): Linear(in_features=384, out_features=1152, bias=True)
        (q_norm): Identity()
        (k_norm): Identity()
        (attn_drop): Dropout(p=0.0, inplace=False)
        (norm): Identity()
        (proj): Linear(in_features=384, out_features=384, bias=True)
        (proj_drop): Dropout(p=0.0, inplace=False)
      )
      (ls1): Identity()
      (drop_path1): Identity()
      (norm2): LayerNorm((384,), eps=1e-06, elementwise_affine=True)
      (mlp): Mlp(
        (fc1): Linear(in_features=384, out_features=1536, bias=True)
        (act): GELU(approximate='none')
        (drop1): Dropout(p=0.0, inplace=False)
  

In [ ]:
#| export
class ViTPretrained(nn.Module):
    def __init__(self, model_name= "vit_small_patch8_224", pretrained= True, num_classes= 10):
        super().__init__()

        self.backbone = timm_vit_model(model_name, pretrained= pretrained)
        self.head = nn.Linear(384, num_classes)

    def forward(self, x):
        h = self.backbone(x)
        logits = self.head(h)
        return logits

In [ ]:
#| hide
model = ViTPretrained()
print(model.backbone(torch.randn(1, 3, 224, 224)).shape)
model(torch.randn(1, 3, 224, 224)).shape

torch.Size([1, 384])


torch.Size([1, 10])

## A unified API for ViT models

In [ ]:
#| export
from typing import Union, Literal
from fedai.models.vision.registery import register_model

In [ ]:
#| export
@register_model(
    name='vit',
    variants=['small', 'base', 'timm_']
)
def make_vit(
    variant: Literal['small', 'base', 'timm_'] = 'small',
    pretrained: bool = False,
    num_classes: int = 10,
    **kwargs
) -> nn.Module:
    """Create a Vision Transformer model."""

    if pretrained and variant == "timm_":
        return ViTPretrained(pretrained=True, num_classes=num_classes)
    
    model_dict = {'small': ViTSmall, 'base': ViT}
    
    if variant not in model_dict:
        raise ValueError(f"Unsupported ViT variant: {variant}. "
                        f"Supported: {list(model_dict.keys())}")
    
    return model_dict[variant](num_classes=num_classes, **kwargs)

In [ ]:
#| hide
kwargs = dict(img_size = (3, 32, 32), patch_size = 4, dim= 512, depth= 6, heads= 8, mlp_dim= 1024)
model = make_vit(**kwargs)
model

ViTSmall(
  (backbone): ViTSmallBackbone(
    (to_patch_embedding): SPT(
      (to_patch_tokens): Sequential(
        (0): Rearrange('b c (h p1) (w p2) -> b (h w) (p1 p2 c)', p1=4, p2=4)
        (1): LayerNorm((240,), eps=1e-05, elementwise_affine=True)
        (2): Linear(in_features=240, out_features=512, bias=True)
      )
    )
    (dropout): Dropout(p=0.0, inplace=False)
    (transformer): Transformer(
      (layers): ModuleList(
        (0-5): 6 x ModuleList(
          (0): PreNorm(
            (norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
            (fn): LSA(
              (attend): Softmax(dim=-1)
              (to_qkv): Linear(in_features=512, out_features=1536, bias=False)
              (to_out): Sequential(
                (0): Linear(in_features=512, out_features=512, bias=True)
                (1): Dropout(p=0.0, inplace=False)
              )
            )
          )
          (1): PreNorm(
            (norm): LayerNorm((512,), eps=1e-05, elementwise_af

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()